# Clerk with a Landscape and Clustergram: Atera human kidney

Scratch notebook (not part of the docs) for a Clerk case on the Atera kidney tutorial data: a non-diseased human kidney FFPE section, whole-transcriptome imaging, 28 default Leiden clusters.

Layout: a Landscape and a Clustergram linked to each other, with Clerk as a sidecar column at the end of the row.

What to try:

1. **Ask.** Explore, then ask Clerk about what you see. Its answers link genes, clusters and the views it cites: click them to show them in the Landscape and Clustergram.
2. **Label cell types.** Clerk records the cell types it proposes and offers to put them on the Clustergram (**Label**). Add or edit your own in the Clustergram's `cell_type` column category (and marker genes in the `info` row category). Clerk never overwrites a label you set.
3. **Save.** **Save** in the panel (or `clerk.save_conversation()`) writes one JSON file; **Report** (or `clerk.export()`) writes a folder with an HTML report and screenshots. Both keep the conversation, its links, Clerk's proposals, and the Clustergram's labels, with which ones Clerk set.
4. **Reopen and browse.** Set `CASE` below to a saved file or folder, then restart and run all. The conversation comes back with working links, its views become pins on the Landscape (**PINS**), and the Clustergram shows the labels again. Reading it needs no Claude; a new question continues the case.

In [1]:
import os

# Use this checkout's JS build (npm run build / npm run watch), served once over HTTP
# so each widget stores a tiny import instead of the whole bundle. After a rebuild,
# restart the kernel and reload the page. If widgets fail to load (e.g. a remote
# kernel), use "1" instead to inline the bundle. Set before importing celldega.
os.environ["CELLDEGA_LOCAL_ESM"] = "serve"

In [2]:
from importlib.metadata import version
from pathlib import Path
from urllib.request import urlopen

import celldega as dega
import numpy as np

print(f"celldega {version('celldega')}")

SAMPLE = "Atera_v1_WTA_Kidney_non_diseased_FFPE"
DATA_REVISION = "3a46ab4db67821eef3b8ff8d1c3704fb7b3bb48f"
# DegaFiles for the Landscape: GitHub (one file per tile).
BASE_URL = f"https://raw.githubusercontent.com/broadinstitute/Celldega_{SAMPLE}/{DATA_REVISION}"

# This dataset's Clerk cases (Save, Report) and Enrichr cache, next to this notebook.
CLERK_FILES = Path("clerk_files/atera_kidney")

# A saved case to reopen: a conversation from Save (.json), a report folder from
# Report, or a folder of them (CLERK_FILES opens the most recent). None starts a new
# conversation.
CASE = None

# Atera centroids are in microns; this matrix maps microns to image pixels.
with urlopen(f"{BASE_URL}/micron_to_image_transform.csv") as response:
    transform = np.loadtxt(response)

celldega 0.28.0a1


## Log in to Claude (one time)

Clerk sends questions to Claude through the `claude` command-line tool (Claude Code), with your own Claude account. If you haven't yet, run `claude auth login` in a terminal on the machine running this kernel. Celldega never sees your credentials. Reopening a saved case doesn't need Claude.

In [3]:
status = dega.clerk.claude_auth_status()
if status["logged_in"]:
    plan = f", {status['subscription_type']} plan" if status["subscription_type"] else ""
    print(f"Logged in to Claude ({status['auth_method']}{plan}).")
else:
    print("Not logged in yet:", status.get("error") or "run `claude auth login` in a terminal.")

Logged in to Claude (claude.ai, enterprise plan).


## Cluster signatures (computed once)

The Clustergram shows the default clusters' pseudo-bulk signatures, as in the [Atera kidney tutorial](https://broadinstitute.github.io/celldega/examples/tutorial_notebooks/Atera_Kidney_Landscape-Clustergram/): dot colors are log1p CPM z-scored per gene, dot sizes the fraction of cells expressing, and markers come from a t-test on up to 2,000 sampled cells per cluster. Each gene is labeled with the cluster where it ranks highest (`leiden_marker`).

The clustered Matrix is cached in `notebooks/data/atera_data` and read with `Matrix.read_zarr`. Without the cache, it's computed once from the cell-level AnnData (a 1.4 GB download the first time). The Landscape doesn't need the AnnData: it reads the default clusters from the DegaFiles.

In [4]:
# Data in the repository's notebooks/data folder when run from a checkout,
# otherwise beside the notebook.
cwd = Path.cwd().resolve()
repo = next((p for p in [cwd, *cwd.parents] if (p / "pyproject.toml").is_file()), None)
data_dir = (repo / "notebooks/data/atera_data" if repo else cwd / "data") / SAMPLE
data_dir.mkdir(parents=True, exist_ok=True)
matrix_store = data_dir / "kidney_leiden_marker_matrix.zarr"
ANNDATA_URL = (
    "https://huggingface.co/datasets/broadinstitute/Celldega_Supporting_Data/resolve/main/"
    f"{SAMPLE}.h5ad"
)
MIN_CELLS = 100
SEED = 0


def compute_matrix():
    import anndata as ad
    import pandas as pd
    import scanpy as sc

    source = sc.read(data_dir / f"{SAMPLE}.h5ad", backed="r", backup_url=ANNDATA_URL)
    counts = source.obs["leiden"].value_counts()
    cells = source[source.obs["leiden"].isin(counts[counts >= MIN_CELLS].index)].to_memory()
    source.file.close()

    signatures = sc.get.aggregate(cells, by="leiden", func=["sum", "count_nonzero"])
    signatures.obs["n_cells"] = cells.obs["leiden"].value_counts().reindex(signatures.obs_names)
    signatures.X = signatures.layers["sum"].astype(np.float32)
    signatures.layers["fraction_expressing"] = (
        signatures.layers["count_nonzero"] / signatures.obs["n_cells"].to_numpy()[:, None]
    )
    sc.pp.normalize_total(signatures, target_sum=1e6)
    sc.pp.log1p(signatures)
    palette = pd.Series(cells.uns["leiden_colors"], index=cells.obs["leiden"].cat.categories)
    signatures.obs["color"] = signatures.obs_names.map(palette)
    signatures.uns["leiden_colors"] = palette.loc[signatures.obs_names].tolist()

    # Rank markers on log-normalized cells, sampling up to 2,000 per cluster.
    rng = np.random.default_rng(SEED)
    sampled = np.concatenate([
        rng.choice(indices, min(len(indices), 2_000), replace=False)
        for indices in cells.obs.groupby("leiden", observed=True).indices.values()
    ])
    ranked = ad.AnnData(
        cells.X[sampled].astype(np.float32),
        obs=cells.obs.iloc[sampled][["leiden"]].copy(),
        var=pd.DataFrame(index=cells.var_names),
    )
    sc.pp.normalize_total(ranked, target_sum=10_000)
    sc.pp.log1p(ranked)
    sc.tl.rank_genes_groups(ranked, groupby="leiden", method="t-test", n_genes=ranked.n_vars)
    signatures.uns["rank_genes_groups"] = ranked.uns["rank_genes_groups"]
    markers = sc.get.rank_genes_groups_df(ranked, group=None)
    best_group = (
        markers.sort_values("scores", ascending=False)
        .drop_duplicates("names")
        .set_index("names")["group"]
    )
    signatures.var["leiden_marker"] = pd.Categorical(
        best_group.reindex(signatures.var_names), categories=cells.obs["leiden"].cat.categories,
    )

    mat = dega.clust.Matrix(
        signatures, col_attr=["leiden"], row_attr=["leiden_marker"],
        col_entity={"entity": "cell", "attr": "leiden"},
    )
    mat.set_size_matrix(ad.AnnData(
        signatures.layers["fraction_expressing"], obs=signatures.obs.copy(), var=signatures.var.copy(),
    ))
    mat.norm("row", by="zscore")
    mat.cluster(view="rank_genes_groups", levels=[1, 2, 3, 5, 10, 25, 50, 75])
    mat.write_zarr(matrix_store)
    return mat


if matrix_store.exists():
    matrix = dega.clust.Matrix.read_zarr(matrix_store)
    print(f"Read the clustered Matrix from {matrix_store}")
else:
    print("Computing the clustered Matrix (a few minutes the first time).")
    matrix = compute_matrix()
print(f"{matrix.to_df().shape[1]} clusters x {matrix.to_df().shape[0]:,} genes")

Read the clustered Matrix from /Users/feni/Documents/celldega/notebooks/data/atera_data/Atera_v1_WTA_Kidney_non_diseased_FFPE/kidney_leiden_marker_matrix.zarr
28 clusters x 18,049 genes


## Landscape, Clustergram and Clerk

The Clustergram's `cell_type` column category holds cell types, yours and Clerk's, and its `info` row category the marker genes behind them. Click a gene row to color the cells by it, or a cluster column to select its cells.

Some neutral questions to start with:

- *"Propose tentative cell types for all clusters."*
- *"What's in this field of view?"* (pan and zoom first)
- *"Do any clusters look like a mixture of two cell types, or a segmentation artifact?"*
- *"Which clusters are hard to tell apart, and what would separate them?"

In [5]:
landscape = dega.viz.Landscape(
    technology="Xenium", base_url=BASE_URL, transform=transform,
    dataset_name="Atera kidney — default clusters", height=600, max_tiles_to_view=25,
)
cgm = dega.viz.Clustergram(
    matrix=matrix, width=400, height=500, rank_dim=5,
    manual_col_cat="cell_type", manual_row_cat="info",
)
clerk = dega.viz.Clerk(
    landscape=landscape,
    clustergram=cgm,
    # Describe the sample, not the answers.
    context_info=(
        "Human kidney, non-diseased FFPE section, whole-transcriptome imaging-based "
        "spatial transcriptomics. Cells were segmented and Leiden-clustered "
        "(28 clusters with at least 100 cells). The Landscape shows the tissue image with "
        "cells colored by Leiden cluster."
    ),
    model="",  # "" = your CLI default; "opus", "sonnet" or "haiku"
)
clerk.output_dir = str(CLERK_FILES)
if CASE:
    # The conversation, Clerk's proposals and the Clustergram's labels come back;
    # its views become pins on the Landscape. New saves go beside it.
    case = clerk.open_case(CASE)
    print(f"Reopened {case['path']} (saved {case.get('saved')})")

In [6]:
# The Landscape and Clustergram keep their widths and Clerk takes the room left
# (240-300 px). A row wider than the notebook scrolls sideways; or put Clerk full
# width underneath with clerk_layout="wide".
linked = dega.viz.spatial_clustergram(landscape, cgm, width="500px", clerk=clerk)

In [8]:
linked

## Cell types so far

One row per cluster labeled on the Clustergram or proposed by Clerk: the label (`set_by` you or Clerk) and Clerk's latest proposal, whether or not it's on the Clustergram. Re-run after labeling.

In [ ]:
clerk.cell_types()

## Save the case

**Save** in the panel does the same as the first line below: one JSON file in `clerk_files/atera_kidney/conversations/`. **Report** (or `clerk.export()`) writes a folder in `clerk_files/atera_kidney/` with `report.html`, `report.ipynb`, `record.json`, the screenshots, `findings.csv` and `cluster_labels.csv` (the table above). A name already taken gets `-2`, `-3`, ...

Ask Clerk a question or two first.

In [ ]:
if clerk.record:
    saved = clerk.save_conversation(title="atera kidney")
    report = clerk.export(title="Atera kidney with Clerk")
    print("Conversation:", saved)
    print("Report:", report["record"].parent)
else:
    print("Ask Clerk a question first.")

## Reopen it

In this session: the cell below reopens the saved conversation in the panel above, replacing what's there. After restarting the kernel: set `CASE` at the top (e.g. `CASE = CLERK_FILES` for the most recent case, or the path printed above) and run all.

Then browse the case: click its genes, clusters and views; open **PINS** on the Landscape to revisit each question's view (hover a pin for its question); and check that the Clustergram shows the cell types again.

In [ ]:
# clerk.open_case(saved)  # or a report folder, or CLERK_FILES for the most recent

In [ ]:
# Clerk's proposals, with their evidence:
clerk.findings_df()

In [ ]:
# Each question's views as Snapshot cards (e.g. clerk.snapshots.card("Q1 Landscape")):
clerk.snapshots